# Natural-Language Data Preparation: Split the Bookings

This notebook pairs a plain-language request with the code that carries it out. It makes the arrival-date split only. It does not clean data, select or engineer features, or train a model.

## Request in plain language

> Load the complete bookings file without changing its rows or values. Add the arrival date needed by the evaluation contract, then split every booking into train, choose, or proof using the project periods. Show the row counts and date ranges. Do not inspect cancellation outcomes in the proof period.

The contract periods are: **train** before 2016-07-01; **choose** from 2016-07-01 through 2016-12-31; **proof** from 2017-01-01 onward.

The split DataFrames stay in memory as `train`, `choose`, and `proof`; this notebook does not write duplicate CSV files.


## 1. Find the project and import the supplied data/contract helpers

The path search lets this notebook run from the repository folder or from `notebooks/`.

In [1]:
from pathlib import Path
import sys

# Look in this folder and its parent folders for the project source code.
repo_root = None
for folder in (Path.cwd(), *Path.cwd().parents):
    if (folder / "src" / "data.py").exists():
        repo_root = folder
        break

if repo_root is None:
    raise FileNotFoundError("Could not find the project src/ folder.")

source_folder = str(repo_root / "src")
if source_folder not in sys.path:
    sys.path.insert(0, source_folder)

import contract
import data
import pandas as pd

print(f"Project folder: {repo_root}")

Project folder: C:\Users\Lenovo\Desktop\Build_2_Repo\IIT_GN_W06


## 2. Load the raw bookings and add the arrival date

`read_raw()` preserves the source rows and values. We add the derived arrival-date column to a separate DataFrame for splitting.

In [2]:
# Load the source exactly as provided.
raw_bookings = data.read_raw()

# Keep the raw DataFrame unchanged; add the date to a working copy.
bookings = raw_bookings.copy()
bookings["arrival_date"] = data.arrival_date(bookings)

print(f"Raw rows: {len(raw_bookings):,}")
print(f"Raw columns: {raw_bookings.shape[1]}")
print(f"Arrival-date rows: {len(bookings):,}")
print(f"First arrival date: {bookings['arrival_date'].min().date()}")
print(f"Last arrival date: {bookings['arrival_date'].max().date()}")

Raw rows: 119,390
Raw columns: 32
Arrival-date rows: 119,390
First arrival date: 2015-07-01
Last arrival date: 2017-08-31


## 3. Split by the project arrival-date periods

Use `contract.split_periods()` so the notebook follows the supplied scoring contract. No random split is used.

In [3]:
train, choose, proof = contract.split_periods(bookings)

print(f"Train rows: {len(train):,}")
print(f"Choose rows: {len(choose):,}")
print(f"Proof rows: {len(proof):,}")
print(f"Rows in all periods: {len(train) + len(choose) + len(proof):,}")
print(f"Rows in source: {len(bookings):,}")

Train rows: 49,157
Choose rows: 29,546
Proof rows: 40,687
Rows in all periods: 119,390
Rows in source: 119,390


## 4. Review each split

This summary uses only row counts, arrival dates, and hotel names. It does not calculate cancellation rates or inspect proof outcomes.

In [4]:
def summarize_period(name, period):
    """Show the size and date range of one period without inspecting outcomes."""
    return {
        "period": name,
        "rows": len(period),
        "first_arrival": period["arrival_date"].min().date(),
        "last_arrival": period["arrival_date"].max().date(),
        "hotels": sorted(period["hotel"].dropna().unique()),
    }

period_summary = pd.DataFrame([
    summarize_period("train", train),
    summarize_period("choose", choose),
    summarize_period("proof", proof),
])

print(period_summary.to_string(index=False))

period  rows first_arrival last_arrival                     hotels
 train 49157    2015-07-01   2016-06-30 [City Hotel, Resort Hotel]
choose 29546    2016-07-01   2016-12-31 [City Hotel, Resort Hotel]
 proof 40687    2017-01-01   2017-08-31 [City Hotel, Resort Hotel]


## What this step establishes

- Every source booking is assigned to one of the contract periods.
- The period DataFrames are available for the next data-quality and EDA work.
- No cleaning, feature selection, feature engineering, or modeling has happened yet.
- Proof cancellation outcomes have not been summarized. Keep them reserved until method and settings are frozen.

Next, inspect data quality and distributions on train, document any cleaning or retention decisions, and decide which timely fields can become features.